# Turkey Hazelnut Production Models

**Target:** FAOSTAT Turkey hazelnut production YoY log-return (`prod_ret`)  
**Sample:** 1962–2024, n=63 (frost back to 1940; precip only 1990–2024, mean-filled for earlier years)

**Methodology:** Relaxed Lasso→Ridge — LassoCV selects features, RidgeCV refits on survivors to remove shrinkage bias. Same approach as `scripts/production_regression.py`.

| Model | Spec | R²(IS) | CV R² | n | Notes |
|-------|------|--------|-------|---|-------|
| **M1** | `prod_ret ~ prod_ret_lag1` | 0.517 | — | 63 | Biennial bearing — dominant signal |
| **M2** | Lasso→Ridge: lag1 + frost + precip | 0.571 | 0.409 | 63 | **Primary** — script output |
| **M3** | Lasso→Ridge: frost + precip only (no lag) | tbd | tbd | 63 | Weather-only trigger |

**Selected features (M2):** `prod_ret_lag1`, `emarch_dh`, `april_dh`, `aug_mm`  
**Dropped by Lasso:** `feb_dh`, `lmarch_dh`, `may_dh`, `frost_dh`, `pollin_mm`, `summer_mm`

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LassoCV, RidgeCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
DATA = Path('../data/raw')

In [8]:
fuck = pd.read_csv("../data/raw/era5_frost_monthly.csv")
# fuck.head()
fuck[fuck["year"] == 2015]
len(fuck)

85

In [ ]:
# --- Load ---
fao    = pd.read_csv(DATA / 'faostat/turkey_hazelnut_production.csv', index_col='year')
frost  = pd.read_csv(DATA / 'era5_frost_monthly.csv',  index_col='year')
precip = pd.read_csv(DATA / 'era5_precip_monthly.csv', index_col='year')

prod         = fao['production_mt'].dropna()
prod_ret     = np.log(prod / prod.shift(1)).rename('prod_ret')
prod_ret_lag = prod_ret.shift(1).rename('prod_ret_lag1')

# Build feature matrix — drop cols >50% missing, mean-fill remainder (matches production_regression.py)
X_raw    = pd.concat([prod_ret_lag, frost, precip], axis=1)
combined = pd.concat([prod_ret, X_raw], axis=1).dropna(subset=['prod_ret'])
combined = combined.dropna(axis=1, thresh=len(combined) // 2)

FEAT_COLS = [c for c in combined.columns if c != 'prod_ret']
for col in FEAT_COLS:
    if combined[col].isna().any():
        combined[col] = combined[col].fillna(combined[col].mean())

FROST_COLS = [c for c in FEAT_COLS if c not in ('prod_ret_lag1', 'pollin_mm', 'summer_mm', 'aug_mm')]

print(f'n={len(combined)}  ({combined.index.min()}–{combined.index.max()})')
print(f'Features ({len(FEAT_COLS)}): {FEAT_COLS}')

## Feature correlations with `prod_ret`

In [ ]:
corr_rows = []
for col in FEAT_COLS:
    if combined[col].std() == 0: continue
    r, p = stats.pearsonr(combined['prod_ret'], combined[col])
    corr_rows.append({'feature': col, 'r': round(r, 3), 'R²': round(r**2, 3), 'p': round(p, 3)})

corr_df = pd.DataFrame(corr_rows).sort_values('R²', ascending=False).set_index('feature')
corr_df

In [ ]:
valid = corr_df.dropna()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in valid['r'][::-1]]
ax.barh(valid.index[::-1], valid['r'][::-1], color=colors, edgecolor='none')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson r  (vs annual production log-return)')
ax.set_title('Feature correlations — biennial lag dominates; emarch_dh and april_dh have signal')
plt.tight_layout()
plt.show()

## M1 — Biennial bearing: `prod_ret ~ prod_ret_lag1`

Hazelnut trees alternate heavy/light crop years — a large crop depletes reserves, suppressing the following year. This is the single strongest predictor.

In [ ]:
d1 = combined[['prod_ret', 'prod_ret_lag1']].dropna()
m1 = sm.OLS(d1['prod_ret'], sm.add_constant(d1[['prod_ret_lag1']])).fit()
print(m1.summary2())

## M2 — Relaxed Lasso→Ridge: full feature set  *(primary model)*

Lasso selects from all candidates; Ridge refits on survivors. Matches `scripts/production_regression.py`.

In [ ]:
def relaxed_lasso_ridge(df, feat_cols, label=''):
    y = df['prod_ret'].values
    X = df[feat_cols].values
    n = len(y)
    cv = min(5, n // 6)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    lasso = LassoCV(alphas=np.logspace(-4, 2, 300), cv=cv, max_iter=50000, n_jobs=-1)
    lasso.fit(Xs, y)
    mask = np.abs(lasso.coef_) > 1e-6

    if mask.sum() == 0:
        lasso.alpha_ = lasso.alphas_[len(lasso.alphas_) // 4]
        lasso.coef_  = Lasso(alpha=lasso.alpha_, max_iter=50000).fit(Xs, y).coef_
        mask = np.abs(lasso.coef_) > 1e-6

    sel = [feat_cols[i] for i in range(len(feat_cols)) if mask[i]]

    ridge = RidgeCV(alphas=np.logspace(-3, 4, 200), cv=cv)
    ridge.fit(Xs[:, mask], y)
    yhat  = ridge.predict(Xs[:, mask])
    r2_is = 1 - np.sum((y - yhat)**2) / np.sum((y - y.mean())**2)
    cv_r2 = cross_val_score(ridge, Xs[:, mask], y, cv=cv, scoring='r2').mean()

    print(f'{label}  Lasso α={lasso.alpha_:.5f}  selected {len(sel)}/{len(feat_cols)}: {sel}')
    print(f'  R²(IS)={r2_is:.3f}   CV R²={cv_r2:.3f}   n={n}')
    coef_df = pd.DataFrame({'feature': sel, 'ridge_coef (std)': ridge.coef_.round(4)})
    print(coef_df.to_string(index=False))
    return dict(sel=sel, yhat=yhat, r2_is=r2_is, cv_r2=cv_r2, scaler=scaler,
                mask=mask, lasso=lasso, ridge=ridge, feat_cols=feat_cols, y=y, Xs=Xs)

res2 = relaxed_lasso_ridge(combined, FEAT_COLS, label='M2 (full)')

## M3 — Relaxed Lasso→Ridge: weather only (no biennial lag)

What does the weather signal look like on its own? Useful for designing a pure weather trigger — i.e. payouts driven by frost/precip irrespective of biennial cycle.

In [ ]:
weather_feats = [c for c in FEAT_COLS if c != 'prod_ret_lag1']
res3 = relaxed_lasso_ridge(combined, weather_feats, label='M3 (weather only)')

## Comparison

In [ ]:
cmp = pd.DataFrame([
    {'': 'M1', 'Spec': 'prod_ret ~ prod_ret_lag1 (OLS)',
     'R²(IS)': round(m1.rsquared, 3), 'CV R²': '—', 'n': int(m1.nobs),
     'Selected': 'prod_ret_lag1', 'Note': 'biennial bearing only'},
    {'': 'M2', 'Spec': 'Lasso→Ridge — full (lag + frost + precip)',
     'R²(IS)': round(res2['r2_is'], 3), 'CV R²': round(res2['cv_r2'], 3), 'n': len(combined),
     'Selected': ', '.join(res2['sel']), 'Note': 'primary — matches production_regression.py'},
    {'': 'M3', 'Spec': 'Lasso→Ridge — weather only (frost + precip)',
     'R²(IS)': round(res3['r2_is'], 3), 'CV R²': round(res3['cv_r2'], 3), 'n': len(combined),
     'Selected': ', '.join(res3['sel']), 'Note': 'pure weather trigger'},
]).set_index('')
cmp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# M1: biennial scatter
ax = axes[0]
ax.scatter(d1['prod_ret_lag1'], d1['prod_ret'], s=30, c='steelblue', zorder=3)
xs = np.linspace(d1['prod_ret_lag1'].min(), d1['prod_ret_lag1'].max(), 100)
p  = m1.params
ax.plot(xs, p['const'] + p['prod_ret_lag1'] * xs, 'k--', lw=1.2)
ax.axhline(0, color='grey', lw=0.5, ls='--'); ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('prod_ret (t−1)'); ax.set_ylabel('prod_ret (t)')
ax.set_title(f'M1 — Biennial  R²={m1.rsquared:.3f}')

# M2: actual vs fitted time series
ax = axes[1]
yrs = combined.index
ax.plot(yrs, combined['prod_ret'], 'k-', lw=1.2, marker='o', ms=3, label='Actual')
ax.plot(yrs, res2['yhat'], 'b--', lw=1.0, label=f'M2 full (R²={res2["r2_is"]:.3f})')
ax.plot(yrs, res3['yhat'], 'r--', lw=1.0, alpha=0.7, label=f'M3 weather (R²={res3["r2_is"]:.3f})')
ax.axhline(0, color='grey', lw=0.5)
ax.set_ylabel('Production log-return')
ax.set_title('Actual vs M2 (full) and M3 (weather only)')
ax.legend(fontsize=8)

# M2: residuals
ax = axes[2]
resid2 = combined['prod_ret'].values - res2['yhat']
colors = ['#e74c3c' if r < 0 else '#2ecc71' for r in resid2]
ax.bar(yrs, resid2, color=colors, edgecolor='none', width=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Actual − Predicted log-return')
ax.set_title(f'M2 residuals  σ={resid2.std():.3f}')

plt.tight_layout()
plt.show()